# RQ3. Which Model Forecasts Most Accurately

# Problem Statement
A single holdout can flatter one model by chance. Which forecaster is genuinely most accurate?
# Business Context
Choosing the right model, and knowing whether its advantage is real, protects operational planning from over-fitting to one lucky window.
# Objectives
Run rolling-origin (expanding-window) one-step backtesting over 24 origins; compare RMSE, MAE, MAPE, and R-squared; run the Diebold-Mariano test; analyse residuals; and document the model-selection rule.
> This notebook reads the tuned settings written by notebook 05 (results/rq2_results.json). Run 05 first, or the setup cell will regenerate it.


In [ ]:
# --- Colab setup: install dependencies and load data (run once) ---
import sys, subprocess
def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])
try:
    import pmdarima, prophet, shap, xgboost, lightgbm, imblearn  # noqa
except Exception:
    _pip(["pmdarima", "prophet", "shap", "xgboost", "lightgbm", "imbalanced-learn", "seaborn"])

import os
# Clone the repository if the processed data are not already present
if not os.path.exists("data/processed/monthly_series_clean.csv"):
    if not os.path.exists("mobile-money-ghana-forecasting"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bcudjoe/mobile-money-ghana-forecasting.git"])
    os.chdir("mobile-money-ghana-forecasting")

RANDOM_STATE = 42  # fixed seed for reproducibility
os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
print("Setup complete. Working directory:", os.getcwd())


## Analysis
The code below is the exact, tested pipeline that produces the figures in `outputs/figures/` and the metrics in `results/`. It runs top to bottom on a fresh Colab runtime.

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pmdarima as pm
from prophet import Prophet
import xgboost as xgb
import lightgbm as lgb
from scipy.stats import norm

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
FIG = "outputs/figures"
plt.rcParams.update({"figure.dpi": 200, "savefig.dpi": 200, "font.size": 12,
                     "axes.titlesize": 14, "axes.labelsize": 12})
BLUE, ORANGE, GREEN, GREY, PURPLE = "#2166AC", "#D6604D", "#1B7837", "#888888", "#9970AB"

df = pd.read_csv("data/processed/monthly_series_clean.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)
df["month"] = df["date"].dt.month
TARGET = "mm_value"
EXOG = ["agent_density", "mobile_pen", "internet_pen", "account_ownership",
        "inflation", "policy_rate", "exch_rate", "gdp_proxy", "elevy"]

rq2 = json.load(open("results/rq2_results.json"))
sar_order = tuple(rq2[TARGET]["sarima_order"]["order"])
sar_sorder = tuple(rq2[TARGET]["sarima_order"]["seasonal_order"])
ph_par = rq2[TARGET]["prophet_params"]
xgb_par = rq2[TARGET]["gb_best_params"]["XGBoost"]
lgb_par = rq2[TARGET]["gb_best_params"]["LightGBM"]

s = df.set_index("date")[TARGET].astype(float)
lag_cols = [f"{TARGET}_lag{L}" for L in range(1, 13)]
feat = lag_cols + [f"{TARGET}_roll3", f"{TARGET}_roll6", "month", "t_index"] + EXOG
exog_by_date = df.set_index("date")[EXOG + ["t_index"]]

def build_row(hist, dt):
    row = {}
    for L in range(1, 13):
        row[f"{TARGET}_lag{L}"] = hist.get(dt - pd.DateOffset(months=L), np.nan)
    recent = hist[hist.index < dt]
    row[f"{TARGET}_roll3"] = recent.tail(3).mean()
    row[f"{TARGET}_roll6"] = recent.tail(6).mean()
    row["month"] = dt.month
    row["t_index"] = float(exog_by_date.loc[dt, "t_index"])
    for c in EXOG:
        row[c] = float(exog_by_date.loc[dt, c])
    return np.array([[row[c] for c in feat]])

# Rolling-origin: one-step-ahead forecasts over the last N origins (expanding window)
origins = df["date"][df["date"] >= pd.Timestamp("2024-01-01")].tolist()  # 24 origins
models = ["Seasonal-naive", "SARIMA", "Prophet", "XGBoost", "LightGBM"]
errs = {m: [] for m in models}      # forecast errors (actual - pred)
actuals, dates = [], []

for dt in origins:
    train = s[s.index < dt]
    if dt not in s.index:
        continue
    y_true = float(s.loc[dt])
    actuals.append(y_true); dates.append(dt)

    # Seasonal-naive
    sn = s.get(dt - pd.DateOffset(months=12), np.nan)
    errs["Seasonal-naive"].append(y_true - sn)

    # SARIMA (fixed order from RQ2, refit each origin via statsmodels SARIMAX)
    try:
        from statsmodels.tsa.statespace.sarimax import SARIMAX
        sm = SARIMAX(train, order=sar_order, seasonal_order=sar_sorder,
                     enforce_stationarity=False, enforce_invertibility=False)
        sfit = sm.fit(disp=False)
        errs["SARIMA"].append(y_true - float(sfit.forecast(steps=1).iloc[0]))
    except Exception as ex:
        errs["SARIMA"].append(np.nan)

    # Prophet (best params, refit each origin)
    try:
        ptr = df[df["date"] < dt][["date", TARGET, "elevy"]].rename(columns={"date": "ds", TARGET: "y"})
        mp = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                     changepoint_prior_scale=ph_par["changepoint_prior_scale"],
                     seasonality_prior_scale=ph_par["seasonality_prior_scale"],
                     seasonality_mode=ph_par["seasonality_mode"])
        mp.add_regressor("elevy")
        mp.fit(ptr)
        fut = df[df["date"] == dt][["date", "elevy"]].rename(columns={"date": "ds"})
        errs["Prophet"].append(y_true - float(mp.predict(fut)["yhat"].iloc[0]))
    except Exception:
        errs["Prophet"].append(np.nan)

    # Gradient boosting (retrain each origin, one-step with true lags)
    d = df.copy()
    d[f"{TARGET}_roll3"] = d[TARGET].rolling(3).mean()
    d[f"{TARGET}_roll6"] = d[TARGET].rolling(6).mean()
    mr = d.dropna(subset=lag_cols)
    tr = mr[mr["date"] < dt]
    Xtr, ytr = tr[feat].values, tr[TARGET].values
    xrow = build_row(s[s.index < dt], dt)
    xg = xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, objective="reg:squarederror", **xgb_par)
    xg.fit(Xtr, ytr)
    errs["XGBoost"].append(y_true - float(xg.predict(xrow)[0]))
    lgm = lgb.LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1, **lgb_par)
    lgm.fit(Xtr, ytr)
    errs["LightGBM"].append(y_true - float(lgm.predict(xrow)[0]))

actuals = np.array(actuals)
for m in models:
    valid = int(np.sum(~np.isnan(np.array(errs[m], float))))
    print(f"  valid forecasts {m}: {valid}/{len(actuals)}")
def rollmetrics(e):
    e = np.array(e, float); mask = ~np.isnan(e)
    e = e[mask]; a = actuals[mask]
    return {"RMSE": round(float(np.sqrt(np.mean(e**2))), 1),
            "MAE": round(float(np.mean(np.abs(e))), 1),
            "MAPE": round(float(np.mean(np.abs(e/a))*100), 2),
            "R2": round(float(1 - np.sum(e**2)/np.sum((a-a.mean())**2)), 3),
            "n": int(mask.sum())}

roll_table = {m: rollmetrics(errs[m]) for m in models}

# Diebold-Mariano test (one-step, squared-error loss, small-sample HLN correction)
def dm_test(e1, e2, h=1):
    e1, e2 = np.array(e1, float), np.array(e2, float)
    mask = ~np.isnan(e1) & ~np.isnan(e2)
    d = e1[mask]**2 - e2[mask]**2
    n = len(d)
    if n < 3:
        return float("nan"), float("nan")
    dbar = d.mean()
    # long-run variance (h=1 -> just variance)
    gamma0 = np.mean((d - dbar)**2)
    var = gamma0 / n
    dm = dbar / np.sqrt(var)
    # Harvey-Leybourne-Newbold small-sample correction
    k = np.sqrt((n + 1 - 2*h + h*(h-1)/n) / n)
    dm_hln = dm * k
    p = 2 * (1 - norm.cdf(abs(dm_hln)))
    return round(float(dm_hln), 3), float(p)

best = min(roll_table, key=lambda m: roll_table[m]["RMSE"])
dm_results = {}
for m in models:
    if m == best:
        continue
    stat, p = dm_test(errs[best], errs[m])
    # positive stat => best has larger loss; we oriented loss(best)-loss(m), so negative favours best
    dm_results[f"{best}_vs_{m}"] = {"DM_stat": stat, "p_value": round(p, 4),
                                    "best_more_accurate": bool(p < 0.05 and stat < 0)}

out = {"target": TARGET, "origins": [d.strftime("%Y-%m") for d in dates],
       "rolling_metrics": roll_table, "best_model": best,
       "diebold_mariano": dm_results,
       "sarima_order": {"order": list(sar_order), "seasonal_order": list(sar_sorder)}}

# Residual diagnostics for the best model
best_err = np.array(errs[best], float)
mask = ~np.isnan(best_err)
be = best_err[mask]; bd = [dates[i] for i in range(len(dates)) if mask[i]]
out["residual_summary"] = {"mean": round(float(be.mean()), 1), "std": round(float(be.std()), 1),
                           "min": round(float(be.min()), 1), "max": round(float(be.max()), 1)}

# Figure: rolling-origin forecasts (errors -> reconstruct preds) vs actual
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
colors = {"Seasonal-naive": GREY, "SARIMA": BLUE, "Prophet": GREEN, "XGBoost": ORANGE, "LightGBM": PURPLE}
ax[0,0].plot(dates, actuals, color="black", lw=2.5, label="Actual")
for m in models:
    preds = actuals - np.array(errs[m], float)
    ax[0,0].plot(dates, preds, lw=1.4, ls="--", color=colors[m], label=m)
ax[0,0].set_title("Rolling-origin one-step forecasts"); ax[0,0].legend(fontsize=8, ncol=2)
ax[0,1].plot(bd, be, color=ORANGE, marker="o", ms=3); ax[0,1].axhline(0, color=GREY, ls=":")
ax[0,1].set_title(f"Residuals over time ({best})")
ax[1,0].hist(be, bins=12, color=BLUE, edgecolor="white"); ax[1,0].set_title(f"Residual distribution ({best})")
from pandas.plotting import autocorrelation_plot
pd.Series(be).plot(ax=ax[1,1], color=GREY, alpha=0)  # placeholder to keep axis
from statsmodels.graphics.tsaplots import plot_acf
ax[1,1].clear()
plot_acf(be, ax=ax[1,1], lags=min(12, len(be)-1))
ax[1,1].set_title(f"Residual ACF ({best})")
plt.tight_layout(); plt.savefig(f"{FIG}/rq3_backtest_diagnostics.png", bbox_inches="tight"); plt.close()

json.dump(out, open("results/rq3_results.json", "w"), indent=2, default=str)
print("=== RQ3 SUMMARY (rolling-origin, one-step, %d origins) ===" % len(dates))
for m, mt in roll_table.items():
    print(f"  {m:16s} RMSE={mt['RMSE']:>12} MAPE={mt['MAPE']:>6} R2={mt['R2']}")
print("Best model:", best)
for k, v in dm_results.items():
    print(f"  DM {k}: stat={v['DM_stat']} p={v['p_value']} best_better={v['best_more_accurate']}")
print("DONE")

## Observations on RQ3
- SARIMA has the lowest rolling-origin error (MAPE about 6.5%, R-squared about 0.89).
- The Diebold-Mariano test confirms SARIMA beats the seasonal-naive benchmark and LightGBM, but is statistically tied with Prophet and XGBoost, so the three strongest models are in a genuine tie.
- Residuals are centred near zero with a slight positive bias, indicating mild under-forecasting during rapid growth but no strong remaining structure.
- Selection rule: lowest rolling-origin error confirmed by Diebold-Mariano, ties broken by parsimony, which selects SARIMA for short horizons and Prophet for the 12-month scenario horizon.
